In [1]:
print("Check_Python")

Check_Python


## Model Training Pipeline

The complete workflow of the model training process is:

1. Load dataset (Normal + Pneumonia images)
2. Preprocess images (resize, normalize, augment)
3. Create data generators
4. Split data (Normal vs Pneumonia)
5. Apply training strategy:
   - Either standard training
   - Or 3-Fold cross validation
6. Train CNN model
8. Save model

In [2]:
import os
from pathlib import Path
import random

import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.utils import load_img
from tensorflow.keras.utils import img_to_array

In [3]:
print(tf.__version__)
print(tf.config.list_physical_devices())

2.20.0
[PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]


## Standard Data Flow (Without Folding)

In the normal training approach:

- Dataset is divided into:
  - Training Set
  - Validation Set
  - Test Set

### Flow:

Dataset → Train Generator → Model → Validation Generator → Metrics

### Explanation:

1. Training generator feeds batches of images to the model.
2. Model learns patterns (Normal vs Pneumonia).
3. Validation generator checks performance after each epoch.
4. Test set is used only after training is complete.

In [4]:
PROJECT_ROOT = Path("..") 
DATA_DIR = PROJECT_ROOT / "data" / "raw"

TRAIN_DIR = DATA_DIR / "train"
VAL_DIR   = DATA_DIR / "val"
TEST_DIR  = DATA_DIR / "test"

TRAIN_DIR, VAL_DIR, TEST_DIR

(WindowsPath('../data/raw/train'),
 WindowsPath('../data/raw/val'),
 WindowsPath('../data/raw/test'))

In [5]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
COLOR_MODE = "grayscale"

In [6]:
#lighter augmentation
train_datagen = ImageDataGenerator(
    rescale=1.0 / 255,
    rotation_range=10,
    zoom_range=0.1,
    horizontal_flip=True
)

We aren't using advanced data augmentation because it will risk the data. problems:-
- large data
- brightness and contrast matters

In [7]:
val_test_datagen = ImageDataGenerator(
    rescale=1.0 / 255
)

A generator does NOT load all images at once.

Instead:
- It loads images in batches (e.g., 32 images)
- Applies preprocessing (resize, normalize)
- Feeds them to the model

Why needed?

- Saves memory
- Enables large dataset training
- Allows augmentation

In [8]:
train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    color_mode=COLOR_MODE,
    class_mode="binary"
)

Found 5216 images belonging to 2 classes.


In [9]:
val_generator = val_test_datagen.flow_from_directory(
    VAL_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    color_mode=COLOR_MODE,
    class_mode="binary"
)

Found 16 images belonging to 2 classes.


In [10]:
#Handling class imbalance
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(train_generator.classes)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_generator.classes
)

class_weights = dict(zip(classes, class_weights))
class_weights

{np.int32(0): np.float64(1.9448173005219984),
 np.int32(1): np.float64(0.6730322580645162)}

The solution we used: Class Weights
* If a class has fewer images → its mistakes are more costly
* If a class has many images → its mistakes cost less

In [11]:
def build_model():
    m = models.Sequential([
        layers.Input(shape=(224, 224, 1)),

        layers.Conv2D(32, (3, 3), activation="relu"),
        layers.MaxPooling2D(2, 2),

        layers.Conv2D(64, (3, 3), activation="relu"),
        layers.MaxPooling2D(2, 2),

        layers.Conv2D(128, (3, 3), activation="relu"),
        layers.MaxPooling2D(2, 2),

        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dense(1, activation="sigmoid")
    ])
    m.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            tf.keras.metrics.Precision(name="precision"),
            tf.keras.metrics.Recall(name="recall")
        ]
    )
    return m

In [12]:
callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    ),
    ModelCheckpoint(
        "best_model_1.h5",
        monitor="val_loss",
        save_best_only=True
    )
]

In [14]:
model = build_model()
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=25,
    class_weight=class_weights,
    callbacks=callbacks
)

Epoch 1/25
163/163 ━━━━━━━━━━━━━━━━━━━━ 0s 582ms/step - accuracy: 0.7196 - loss: 0.5683 - precision: 0.8752 - recall: 0.7329

163/163 ━━━━━━━━━━━━━━━━━━━━ 99s 589ms/step - accuracy: 0.8240 - loss: 0.3725 - precision: 0.9329 - recall: 0.8222 - val_accuracy: 0.6250 - val_loss: 0.8099 - val_precision: 0.5714 - val_recall: 1.0000
Epoch 2/25
163/163 ━━━━━━━━━━━━━━━━━━━━ 0s 523ms/step - accuracy: 0.9251 - loss: 0.1817 - precision: 0.9757 - recall: 0.9219

163/163 ━━━━━━━━━━━━━━━━━━━━ 86s 526ms/step - accuracy: 0.9304 - loss: 0.1757 - precision: 0.9772 - recall: 0.9280 - val_accuracy: 0.6875 - val_loss: 0.5822 - val_precision: 0.6154 - val_recall: 1.0000
Epoch 3/25
163/163 ━━━━━━━━━━━━━━━━━━━━ 0s 516ms/step - accuracy: 0.9409 - loss: 0.1565 - precision: 0.9791 - recall: 0.9408

163/163 ━━━━━━━━━━━━━━━━━━━━ 85s 519ms/step - accuracy: 0.9373 - loss: 0.1682 - precision: 0.9802 - recall: 0.9345 - val_accuracy: 0.9375 - val_loss: 0.2068 - val_precision: 0.8889 - val_recall: 1.0000
Epoch 4/25
163/163 ━━━━━━━━━━━━━━━━━━━━ 0s 519ms/step - accuracy: 0.9463 - loss: 0.1496 - precision: 0.9812 - recall: 0.9457

163/163 ━━━━━━━━━━━━━━━━━━━━ 85s 522ms/step - accuracy: 0.9477 - loss: 0.1386 - precision: 0.9823 - recall: 0.9466 - val_accuracy: 1.0000 - val_loss: 0.0850 - val_precision: 1.0000 - val_recall: 1.0000
Epoch 5/25
163/163 ━━━━━━━━━━━━━━━━━━━━ 89s 543ms/step - accuracy: 0.9417 - loss: 0.1491 - precision: 0.9832 - recall: 0.9375 - val_accuracy: 0.9375 - val_loss: 0.1814 - val_precision: 0.8889 - val_recall: 1.0000
Epoch 6/25
163/163 ━━━━━━━━━━━━━━━━━━━━ 85s 518ms/step - accuracy: 0.9442 - loss: 0.1406 - precision: 0.9825 - recall: 0.9417 - val_accuracy: 0.6250 - val_loss: 0.5518 - val_precision: 0.5714 - val_recall: 1.0000
Epoch 7/25
163/163 ━━━━━━━━━━━━━━━━━━━━ 88s 539ms/step - accuracy: 0.9528 - loss: 0.1284 - precision: 0.9845 - recall: 0.9515 - val_accuracy: 0.9375 - val_loss: 0.1806 - val_precision: 0.8889 - val_recall: 1.0000
Epoch 8/25
163/163 ━━━━━━━━━━━━━━━━━━━━ 83s 509ms/step - accuracy: 0.9542 - loss: 0.1187 - precision: 0.9861 - recall: 0.9517 - val_accuracy: 0.9375 - val_loss

##  3-Fold Cross Validation (Custom Strategy)

Instead of using the entire pneumonia dataset at once:

1. Pneumonia dataset is divided into 3 equal parts:
   - P1, P2, P3

2. Normal dataset remains fixed.

3. We train 3 separate models:

### Fold 1:
Train on: Normal + P1  
Validate on: Remaining data

### Fold 2:
Train on: Normal + P2  
Validate on: Remaining data

### Fold 3:
Train on: Normal + P3  
Validate on: Remaining data

4. Final performance = Average of all folds

In [42]:
pneumonia_dir = TRAIN_DIR / "PNEUMONIA"
normal_dir    = TRAIN_DIR / "NORMAL"

In [43]:
pneumonia_files = sorted(list(pneumonia_dir.glob("*")))
normal_files    = sorted(list(normal_dir.glob("*")))

In [44]:
random.seed(42)
random.shuffle(pneumonia_files)

In [45]:
# Split pneumonia into 3 equal subsets
fold_size = len(pneumonia_files) // 3
pneu_fold1 = pneumonia_files[:fold_size]
pneu_fold2 = pneumonia_files[fold_size : fold_size * 2]
pneu_fold3 = pneumonia_files[fold_size * 2:]

In [46]:
print(f"Total PNEUMONIA : {len(pneumonia_files)}")
print(f"Total NORMAL    : {len(normal_files)}")
print(f"Fold 1 pneu     : {len(pneu_fold1)}")
print(f"Fold 2 pneu     : {len(pneu_fold2)}")
print(f"Fold 3 pneu     : {len(pneu_fold3)}")

Total PNEUMONIA : 3875
Total NORMAL    : 1341
Fold 1 pneu     : 1291
Fold 2 pneu     : 1291
Fold 3 pneu     : 1293


## Generator Behavior

### In Normal Training:
- One train generator
- One validation generator

### In 3-Fold:
- New train generator created for each fold
- Each generator uses different pneumonia subset

So:
Generator changes per fold
Model learns from different data each time

In [51]:
def custom_fold_generator(normal_files, pneumonia_files, batch_size, class_weights, augment=True):
    
    if augment:
        datagen = ImageDataGenerator(
            rotation_range=10,
            zoom_range=0.1,
            horizontal_flip=True
        )
    else:
        datagen = ImageDataGenerator()

    # Combine data
    all_files = list(normal_files) + list(pneumonia_files)
    labels = [0] * len(normal_files) + [1] * len(pneumonia_files)

    while True:

        # Shuffle
        num_samples = len(all_files)
        indices = np.arange(num_samples)
        np.random.shuffle(indices)

        # don't overwrite original lists permanently (important for next epochs)
        shuffled_files = np.array(all_files)[indices]
        shuffled_labels = np.array(labels)[indices]

        # Create batches
        for i in range(0, len(shuffled_files), batch_size):
            
            batch_files = shuffled_files[i:i+batch_size]
            batch_labels = shuffled_labels[i:i+batch_size]

            images = []

            for fpath in batch_files:
                img = load_img(fpath, target_size=IMG_SIZE, color_mode=COLOR_MODE)
                arr = img_to_array(img) / 255.0

                if augment:
                    arr = datagen.random_transform(arr)

                images.append(arr)

            sample_weights = np.array([
                class_weights[int(label)] for label in batch_labels
                ], dtype="float32")

            yield np.array(images), np.array(batch_labels), sample_weights

In [52]:
fold_definitions = [
    (normal_files, pneu_fold1, "fold1"),
    (normal_files, pneu_fold2, "fold2"),
    (normal_files, pneu_fold3, "fold3"),
]

In [53]:
for normal_files, pneu_files, fold_name in fold_definitions:

    #info
    print(f"\nTraining {fold_name}")
    print(f"Normal: {len(normal_files)} | Pneumonia: {len(pneu_files)}")
    print("-" * 80)

    #prepare data & class weights
    y_train = np.array([0] * len(normal_files) + [1] * len(pneu_files))
        
    classes = np.unique(y_train)
    weights = compute_class_weight("balanced", classes=classes, y=y_train)
    class_weights = dict(zip(classes.astype(int), weights))
    
    train_gen = custom_fold_generator(
        normal_files,
        pneu_files,
        batch_size=32,
        class_weights=class_weights,
        augment=True
    )



    print("Class weights:", class_weights)

    #model 
    model = build_model()
    callbacks = [
        EarlyStopping(
            monitor="val_loss",
            patience=5,
            restore_best_weights=True
        ),
        ModelCheckpoint(
            f"best_model_{fold_name}.h5",
            monitor="val_loss",
            save_best_only=True
        )
    ]

    steps_per_epoch = (len(normal_files) + len(pneu_files)) // 32
    
    #train
    model.fit(
        train_gen,
        steps_per_epoch=steps_per_epoch,
        validation_data=val_generator,
        epochs=25,
        # class_weight=class_weights, pyhton generator . Therefore, we will apply this inside generator itself
        callbacks=callbacks
)


Training fold1
Normal: 1341 | Pneumonia: 1291
--------------------------------------------------------------------------------
Class weights: {np.int64(0): np.float64(0.9813571961222968), np.int64(1): np.float64(1.0193648334624321)}
Epoch 1/25
82/82 ━━━━━━━━━━━━━━━━━━━━ 0s 645ms/step - accuracy: 0.6524 - loss: 0.6291 - precision: 0.6470 - recall: 0.6732

82/82 ━━━━━━━━━━━━━━━━━━━━ 56s 661ms/step - accuracy: 0.7633 - loss: 0.4712 - precision: 0.7536 - recall: 0.7683 - val_accuracy: 0.8125 - val_loss: 0.5223 - val_precision: 0.7273 - val_recall: 1.0000
Epoch 2/25
82/82 ━━━━━━━━━━━━━━━━━━━━ 0s 633ms/step - accuracy: 0.8802 - loss: 0.2997 - precision: 0.8710 - recall: 0.9020

82/82 ━━━━━━━━━━━━━━━━━━━━ 52s 640ms/step - accuracy: 0.8985 - loss: 0.2614 - precision: 0.8931 - recall: 0.9015 - val_accuracy: 0.8125 - val_loss: 0.3570 - val_precision: 0.7778 - val_recall: 0.8750
Epoch 3/25
82/82 ━━━━━━━━━━━━━━━━━━━━ 51s 627ms/step - accuracy: 0.9277 - loss: 0.1804 - precision: 0.9307 - recall: 0.9204 - val_accuracy: 0.6875 - val_loss: 0.8553 - val_precision: 0.6154 - val_recall: 1.0000
Epoch 4/25
82/82 ━━━━━━━━━━━━━━━━━━━━ 51s 628ms/step - accuracy: 0.9312 - loss: 0.1881 - precision: 0.9281 - recall: 0.9332 - val_accuracy: 0.8750 - val_loss: 0.3748 - val_precision: 0.8000 - val_recall: 1.0000
Epoch 5/25
82/82 ━━━━━━━━━━━━━━━━━━━━ 51s 618ms/step - accuracy: 0.9415 - loss: 0.1396 - precision: 0.9467 - recall: 0.9316 - val_accuracy: 0.8125 - val_loss: 0.4088 - val_precision: 0.7273 - val_recall: 1.0000
Epoch 6/25
82/82 ━━━━━━━━━━━━━━━━━━━━ 50s 609ms/step - accuracy: 0.9523 - loss: 0.1232 - precision: 0.9545 - recall: 0.9485 - val_accuracy: 0.7500 - val_loss: 0.6863 -

82/82 ━━━━━━━━━━━━━━━━━━━━ 61s 712ms/step - accuracy: 0.7557 - loss: 0.4546 - precision: 0.7567 - recall: 0.7409 - val_accuracy: 0.8125 - val_loss: 0.3705 - val_precision: 0.7273 - val_recall: 1.0000
Epoch 2/25
82/82 ━━━━━━━━━━━━━━━━━━━━ 0s 627ms/step - accuracy: 0.9034 - loss: 0.2521 - precision: 0.9156 - recall: 0.8827

82/82 ━━━━━━━━━━━━━━━━━━━━ 51s 634ms/step - accuracy: 0.9115 - loss: 0.2204 - precision: 0.9189 - recall: 0.8987 - val_accuracy: 0.8750 - val_loss: 0.2440 - val_precision: 0.8750 - val_recall: 0.8750
Epoch 3/25
82/82 ━━━━━━━━━━━━━━━━━━━━ 51s 623ms/step - accuracy: 0.9327 - loss: 0.1913 - precision: 0.9394 - recall: 0.9225 - val_accuracy: 0.7500 - val_loss: 0.5101 - val_precision: 0.6667 - val_recall: 1.0000
Epoch 4/25
82/82 ━━━━━━━━━━━━━━━━━━━━ 51s 624ms/step - accuracy: 0.9438 - loss: 0.1524 - precision: 0.9463 - recall: 0.9389 - val_accuracy: 0.7500 - val_loss: 0.4421 - val_precision: 0.6667 - val_recall: 1.0000
Epoch 5/25
82/82 ━━━━━━━━━━━━━━━━━━━━ 51s 626ms/step - accuracy: 0.9358 - loss: 0.1703 - precision: 0.9357 - recall: 0.9335 - val_accuracy: 0.8125 - val_loss: 0.2582 - val_precision: 0.7778 - val_recall: 0.8750
Epoch 6/25
82/82 ━━━━━━━━━━━━━━━━━━━━ 51s 628ms/step - accuracy: 0.9369 - loss: 0.1630 - precision: 0.9412 - recall: 0.9294 - val_accuracy: 0.7500 - val_loss: 0.4607 -

82/82 ━━━━━━━━━━━━━━━━━━━━ 62s 735ms/step - accuracy: 0.6909 - loss: 0.5401 - precision: 0.7202 - recall: 0.6056 - val_accuracy: 0.8750 - val_loss: 0.3598 - val_precision: 0.8000 - val_recall: 1.0000
Epoch 2/25
82/82 ━━━━━━━━━━━━━━━━━━━━ 51s 626ms/step - accuracy: 0.8997 - loss: 0.2513 - precision: 0.9008 - recall: 0.8944 - val_accuracy: 0.6875 - val_loss: 0.5431 - val_precision: 0.6154 - val_recall: 1.0000
Epoch 3/25
82/82 ━━━━━━━━━━━━━━━━━━━━ 0s 651ms/step - accuracy: 0.9183 - loss: 0.2190 - precision: 0.9225 - recall: 0.9068

82/82 ━━━━━━━━━━━━━━━━━━━━ 54s 659ms/step - accuracy: 0.9189 - loss: 0.2149 - precision: 0.9216 - recall: 0.9122 - val_accuracy: 0.9375 - val_loss: 0.1931 - val_precision: 0.8889 - val_recall: 1.0000
Epoch 4/25
82/82 ━━━━━━━━━━━━━━━━━━━━ 53s 649ms/step - accuracy: 0.9143 - loss: 0.2146 - precision: 0.9165 - recall: 0.9072 - val_accuracy: 0.8125 - val_loss: 0.3640 - val_precision: 0.7273 - val_recall: 1.0000
Epoch 5/25
82/82 ━━━━━━━━━━━━━━━━━━━━ 52s 638ms/step - accuracy: 0.9424 - loss: 0.1714 - precision: 0.9484 - recall: 0.9336 - val_accuracy: 0.6875 - val_loss: 0.6218 - val_precision: 0.6154 - val_recall: 1.0000
Epoch 6/25
82/82 ━━━━━━━━━━━━━━━━━━━━ 53s 649ms/step - accuracy: 0.9477 - loss: 0.1409 - precision: 0.9483 - recall: 0.9453 - val_accuracy: 0.6250 - val_loss: 0.8833 - val_precision: 0.5714 - val_recall: 1.0000
Epoch 7/25
82/82 ━━━━━━━━━━━━━━━━━━━━ 56s 687ms/step - accuracy: 0.9358 - loss: 0.1633 - precision: 0.9368 - recall: 0.9316 - val_accuracy: 0.6250 - val_loss: 0.8244 -

## Summary

- Standard training → single split → faster but less reliable
- 3-Fold training → multiple splits → more reliable
- Generators → handle data efficiently in batches
- Each fold trains a fresh model
- Final accuracy = average of all folds

## Future Improvement 
- Try implementing Pure k-fold
- Augmentation per batch instead of per sample 